# 01. Kaggle platform + independent AWS networking diagnostics

**Accelerator: None** · fresh Kaggle session · Python 3.13 · generated by `scripts/build_kaggle_diagnostic_notebooks.py`

A troubleshooting control only: **no Stokastisk, no CUDA, no LLM, no Enigma connector, no runtime token, no Git clone.** It separates four things that earlier notebooks ran in the same second: (1) idle platform survival, (2) ordinary public HTTPS, (3) the independent AWS diagnostic endpoint, (4) the Railway FastAPI health endpoint, with a survival window after each. Attach the Kaggle Secret `ENIGMA_DIAG_KEY` (a narrow diagnostic key, never an AWS credential) and NO inputs.

**Reading the result:** the AWS table keeps the last acknowledged marker even if the VM disappears. Death during `controls_survival` = ordinary HTTPS is enough; during `aws_survival` = any third-party endpoint; only during `railway_survival` = correlated with the Railway request specifically. Correlation is not a cause: only Kaggle's platform logs can explain a status-code-44 termination.

**Evidence durability.** Every step prints flushed `MARK` and `VERDICT` lines and appends them to an fsynced `markers.jsonl` in a new, never-reused attempt directory. After a complete platform termination the VM's files may be gone (on 2026-11-26 a new session had to re-clone everything), but cell output printed before the failure was kept in the saved notebook. **Missing files do not prove that no crash occurred.** After a fatal session, copy the last `MARK` lines (UTC time, `t` = seconds since the prelude) into the incident report.

Run the cells top to bottom (or *Run all*). Every step ends with `VERDICT <step>: PASS | FAIL | INCONCLUSIVE`; anything but PASS raises and stops the run. Then run the final **evidence export** cell manually.

In [ ]:
# P0. Prelude: environment guard, attempt id, markers and verdicts. Standard library only: no network, no GPU library is imported.
import os, sys, json, time, uuid, platform, shutil, subprocess, re, pathlib
NOTEBOOK = "01_enigma_kaggle_platform_aws_diagnostics"
EXPECT = "cpu"                                   # "cpu" = Accelerator None; "gpu" = GPU T4 x2
assert os.path.isdir("/kaggle/working"), "Kaggle only: refusing to run anywhere else (never on a developer laptop)"


class DiagnosticStop(Exception):
    """Raised by every non-PASS verdict so that 'Run all' stops here. Later cells must never run after a failure."""


HAS_GPU_DEVICE = os.path.exists("/dev/nvidia0")
if EXPECT == "cpu" and HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. A GPU is attached; set Accelerator = None and start a NEW session")
if EXPECT == "gpu" and not HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. No GPU; set Accelerator = GPU T4 x2 and start a NEW session")
assert sys.version_info[:2] == (3, 13), f"expected Python 3.13 (build-5 wheel is cp313), got {platform.python_version()}"
REQUIRE_NO_INPUTS = True                       # notebook 01 must run with NO attached dataset, competition or utility script
_inputs = sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else []
if REQUIRE_NO_INPUTS and _inputs:
    raise DiagnosticStop(f"INCONCLUSIVE: wrong environment. Inputs are attached {_inputs}; remove every input and start a NEW session")
EXPERIMENT = "build5" if EXPECT == "gpu" else "no-sdk"   # build 5 is the ONLY supported Stokastisk artifact; recorded in every MARK

ATTEMPT = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "-" + uuid.uuid4().hex[:8]
ATTEMPT_DIR = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}/{ATTEMPT}"
os.makedirs(ATTEMPT_DIR, exist_ok=False)               # a new directory per attempt: earlier evidence is never overwritten
T0 = time.monotonic()
RESULTS = {}                                           # verdicts of THIS kernel session only (stale outputs can never satisfy a check)


def _read(path):
    try:
        with open(path) as fh:
            return fh.read(65536)
    except OSError:
        return None


def host():
    """Host memory, cgroup v2 memory (current, max, events), shared memory, disk, uptime and load. No environment variables are read."""
    h = {}
    for line in (_read("/proc/meminfo") or "").splitlines():
        k, _, v = line.partition(":")
        if k in ("MemTotal", "MemAvailable", "Shmem"):
            h[k.lower() + "_mib"] = round(int(v.split()[0]) / 1024, 1)
    mib = lambda s: None if s is None or not s.strip().isdigit() else round(int(s) / 2**20, 1)   # noqa: E731 ("max" = unlimited -> None)
    h["cgroup_current_mib"], h["cgroup_max_mib"] = mib(_read("/sys/fs/cgroup/memory.current")), mib(_read("/sys/fs/cgroup/memory.max"))
    h["cgroup_events"] = {k: int(v) for k, _, v in (l.partition(" ") for l in (_read("/sys/fs/cgroup/memory.events") or "").splitlines()) if v.strip().isdigit()}
    for d in ("/kaggle/working", "/tmp", "/dev/shm"):
        try:
            u = shutil.disk_usage(d)
            h["disk_" + d.strip("/").replace("/", "_")] = {"used_gib": round(u.used / 2**30, 2), "free_gib": round(u.free / 2**30, 2)}
        except OSError:
            pass
    up = _read("/proc/uptime")
    h["vm_uptime_s"] = float(up.split()[0]) if up else None
    h["loadavg"] = os.getloadavg()
    return h


def ident():
    return {"pid": os.getpid(), "ppid": os.getppid(), "pgid": os.getpgid(0), "sid": os.getsid(0)}


def mark(name, **fields):
    rec = {"attempt": ATTEMPT, "nb": NOTEBOOK, "experiment": EXPERIMENT, "marker": name, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "t": round(time.monotonic() - T0, 2), **ident(), **host(), **fields}
    line = json.dumps(rec, default=str)
    with open(f"{ATTEMPT_DIR}/markers.jsonl", "a") as fh:
        fh.write(line + "\n"); fh.flush(); os.fsync(fh.fileno())
    print("MARK", line, flush=True)
    return rec


def verdict(step, status, reason, **evidence):
    assert status in ("PASS", "FAIL", "INCONCLUSIVE"), status
    RESULTS[step] = {"status": status, "reason": reason, "attempt": ATTEMPT, **evidence}
    mark("verdict", step=step, status=status, reason=reason, **evidence)
    print(f"VERDICT {step}: {status} | {reason}", flush=True)
    if status != "PASS":
        raise DiagnosticStop(f"{step} {status}: {reason}")


def require(*steps):
    """Every prerequisite must have PASSED in this very session (same attempt id)."""
    bad = [s for s in steps if RESULTS.get(s, {}).get("status") != "PASS" or RESULTS[s].get("attempt") != ATTEMPT]
    if bad:
        raise DiagnosticStop(f"INCONCLUSIVE: prerequisite(s) did not PASS in this session: {bad}. Start a new session and run from the top.")


def observe(step, seconds, every=15.0, sample=None):
    """Survival observation: one MARK every `every` s. PASS only if the full duration elapsed with no cgroup OOM kill and no failed sample check."""
    start, oom0, n, problems = time.monotonic(), host()["cgroup_events"].get("oom_kill"), 0, []
    mark(step + "_begin", seconds=seconds, every=every)
    while time.monotonic() - start < seconds:
        time.sleep(min(every, max(0.0, seconds - (time.monotonic() - start))))
        n += 1
        extra = sample() if sample else {}
        if extra.get("problem"):
            problems.append(extra["problem"])
        mark(step + "_alive", i=n, **extra)
    el, oom1 = time.monotonic() - start, host()["cgroup_events"].get("oom_kill")
    delta = None if oom0 is None or oom1 is None else oom1 - oom0
    ok = el >= seconds - 1.0 and delta in (None, 0) and not problems
    verdict(step, "PASS" if ok else "FAIL", f"observed {el:.0f}/{seconds}s, oom_kill delta {delta}, problems {problems[:3]}", observed_s=round(el, 1),
            oom_kill_delta=delta, samples=n)


def export_evidence():
    """Zip this attempt's directory (only it). Safe to run after a failure."""
    base = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}-{ATTEMPT}"
    path = shutil.make_archive(base, "zip", ATTEMPT_DIR)
    print("evidence archive:", path, os.path.getsize(path), "bytes", flush=True)
    return path


_os = dict(l.split("=", 1) for l in (_read("/etc/os-release") or "").splitlines() if "=" in l)
_kaggle = {k: os.environ[k] for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_DOCKER_IMAGE") if k in os.environ}   # allow-list only; never the raw environment
mark("prelude", python=platform.python_version(), expect=EXPECT, gpu_device=HAS_GPU_DEVICE, os=_os.get("PRETTY_NAME", "").strip('"'), kaggle=_kaggle, inputs=_inputs)
print("ATTEMPT", ATTEMPT, "| markers:", ATTEMPT_DIR + "/markers.jsonl", flush=True)

In [ ]:
# Independent AWS evidence (Lambda Function URL -> DynamoDB + CloudWatch), NOT Railway and NOT the Enigma database. Optional Kaggle Secret
# ENIGMA_DIAG_KEY (a narrow, revocable diagnostic key; never an AWS credential). rmark() = mark() locally + one bounded event to AWS; it returns True only
# when AWS acknowledged the durable write. Nothing secret, no environment variable and no request body is ever sent.
import urllib.request
DIAG_URL = "https://ygjtcfhrgkhyybl4gotvzmfa440zzlkh.lambda-url.us-east-1.on.aws/"
_DIAG = {"key": None, "seq": 0}
_REMOTE_FIELDS = {"stage", "phase", "http", "latency_s", "target", "exc", "exit_code", "signal", "last_ok", "verdict", "note", "gpu_count",
                  "gpu_mem_used_mib", "gpu_util_pct", "wheel_build", "wheel_sha256", "artifact"}


def _diag_key():
    if _DIAG["key"] is None:
        try:
            from kaggle_secrets import UserSecretsClient
            k = UserSecretsClient().get_secret("ENIGMA_DIAG_KEY").strip()
            _DIAG["key"] = k if re.fullmatch(r"edx_[a-z0-9]{8}_[A-Za-z0-9_-]{43}", k) else ""
        except Exception:
            _DIAG["key"] = ""
    return _DIAG["key"]


def rmark(name, **fields):
    rec = mark(name, **fields)
    key = _diag_key()
    if not key:
        mark("remote_skipped", reason="no ENIGMA_DIAG_KEY Kaggle Secret")
        return False
    _DIAG["seq"] += 1
    ev = {"attempt": ATTEMPT, "nb": NOTEBOOK, "seq": _DIAG["seq"], "marker": name, "utc": rec["utc"], "t": rec["t"], "experiment": EXPERIMENT,
          "pid": rec["pid"], "pgid": rec["pgid"], "sid": rec["sid"], "mem_available_mib": rec.get("memavailable_mib"),
          "cgroup_current_mib": rec.get("cgroup_current_mib"), "cgroup_max_mib": rec.get("cgroup_max_mib"),
          "oom_kill": rec.get("cgroup_events", {}).get("oom_kill"), "vm_uptime_s": rec.get("vm_uptime_s")}
    ev.update({k: v for k, v in fields.items() if k in _REMOTE_FIELDS and v is not None})
    ok = False
    try:
        req = urllib.request.Request(DIAG_URL, data=json.dumps({"events": [ev]}).encode(), method="POST",
                                     headers={"content-type": "application/json", "authorization": f"Bearer {key}"})
        with urllib.request.urlopen(req, timeout=10) as r:
            body = json.loads(r.read(4096))
            ok = r.status == 200 and body.get("stored", 0) + body.get("duplicates", 0) == 1
    except Exception as e:
        mark("remote_failed", seq=_DIAG["seq"], error=type(e).__name__)
    print(f"REMOTE {'ACK' if ok else 'NOT-ACKNOWLEDGED'} {name} seq={_DIAG['seq']}", flush=True)
    return ok

## 01a. Idle survival: 10 minutes, no network at all

In [ ]:
observe("idle_10min", 600, every=15)

## 01b. Ordinary public HTTPS controls, then 2 minutes of survival

In [ ]:
require("idle_10min")
import urllib.request, urllib.error
def get(name, url):
    mark("request_begin", target=name)
    t = time.monotonic(); row = {"target": name}
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Enigma-Kaggle-Diagnostic/4.0", "Accept": "*/*", "Connection": "close"})
        with urllib.request.urlopen(req, timeout=60) as r:
            row["http"] = r.status; r.read(4096)
    except urllib.error.HTTPError as e:
        row["http"] = e.code
    except Exception as e:
        row["exc"] = type(e).__name__
    row["latency_s"] = round(time.monotonic() - t, 3)
    mark("request_end", **row)
    return row
rows = [get("pypi", "https://pypi.org/simple/pip/"), get("gstatic", "https://www.gstatic.com/generate_204"), get("huggingface", "https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B-Instruct")]
ok = all(r.get("http") in (200, 204) for r in rows)
verdict("controls_https", "PASS" if ok else "INCONCLUSIVE", f"{[(r['target'], r.get('http'), r.get('latency_s')) for r in rows]}")

In [ ]:
require("controls_https")
observe("controls_survival_120s", 120, every=15)

## 01c. Independent AWS diagnostic endpoint (tested BEFORE relying on it), then 2 minutes of survival with a marker every 30 s

In [ ]:
require("controls_survival_120s")
first = rmark("aws_first_event", stage="aws", phase="after", note="first independent AWS event")
if not first:
    verdict("aws_endpoint", "INCONCLUSIVE", "AWS did not acknowledge (attach the ENIGMA_DIAG_KEY Kaggle Secret, Internet ON)")
verdict("aws_endpoint", "PASS", "AWS acknowledged a durable diagnostic write")

In [ ]:
require("aws_endpoint")
observe("aws_survival_120s", 120, every=30, sample=lambda: {"problem": None if rmark("aws_alive", stage="aws_survival") else "aws not acknowledged"})

## 01d. Railway FastAPI health (AWS marker persisted BEFORE the request), then 2 minutes of survival with a marker every 15 s

In [ ]:
require("aws_survival_120s")
import urllib.request, urllib.error
if not rmark("before_railway", stage="railway", phase="before", target="enigma_health"):
    verdict("railway_marker", "INCONCLUSIVE", "AWS did not acknowledge the before-marker: not sending the Railway request")
t = time.monotonic(); row = {}
try:
    req = urllib.request.Request("https://enigma-api-production-c8cffe9d.up.railway.app/api/v1/health", headers={"User-Agent": "Enigma-Kaggle-Diagnostic/4.0", "Accept": "application/json", "Connection": "close"})
    with urllib.request.urlopen(req, timeout=60) as r:
        row["http"] = r.status; r.read(4096)
except urllib.error.HTTPError as e:
    row["http"] = e.code
except Exception as e:
    row["exc"] = type(e).__name__
row["latency_s"] = round(time.monotonic() - t, 3)
rmark("after_railway", stage="railway", phase="after", target="enigma_health", **row)
verdict("railway_health", "PASS" if row.get("http") == 200 else "FAIL", f"{row}")

In [ ]:
require("railway_health")
observe("railway_survival_120s", 120, every=15, sample=lambda: {"problem": None if rmark("railway_alive", stage="railway_survival") else "aws not acknowledged"})
rmark("final", verdict="PASS", note="all four comparisons survived")

## Evidence export (local; the AWS record is the durable copy)

In [ ]:
# Evidence export (run it even after a FAIL / INCONCLUSIVE stop). Review the archive before sharing it; it contains no credential.
mark("export_begin", results={k: v["status"] for k, v in RESULTS.items()})
archive = export_evidence()
print(json.dumps({k: {"status": v["status"], "reason": v["reason"]} for k, v in RESULTS.items()}, indent=1), flush=True)